In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


cwd = Path.cwd()
print(cwd)
root=cwd.parents[1]
pd.set_option("display.max_columns",None)
display(root)

c:\Users\sebas\PycharmProjects\Git\Seb_branch\institutional-roi-analysis\notebooks\national_models


WindowsPath('c:/Users/sebas/PycharmProjects/Git/Seb_branch/institutional-roi-analysis')

### Feature Selection for Explanatory Model

Variables used in the initial prediction model (e.g., credential level, distance, and other structural constraints) were excluded from the explanatory model.

This is because the residuals already represent performance after controlling for these factors. Including them again would introduce circular reasoning and reduce the interpretability of the results.

Instead, the explanatory model focuses on institutional characteristics and program composition variables that were not used in the prediction stage, allowing us to better understand what drives over- and underperformance.

In [1]:
driver_df=pd.read_csv(root/'data'/'raw'/'scorecard'/'raw_national_inst_driver.csv')

NameError: name 'pd' is not defined

In [ ]:
driver_df["has_endowment"] = (
    driver_df["endowment_begin"].notna() &
    driver_df["endowment_end"].notna()
).astype(int)

In [ ]:
driver_df["has_endowment"].describe()

count    6322.000000
mean        0.422018
std         0.493920
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000
Name: has_endowment, dtype: float64

In [ ]:
driver_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6322 entries, 0 to 6321
Data columns (total 49 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   program_percentage_agriculture                           5582 non-null   float64
 1   program_percentage_resources                             5582 non-null   float64
 2   program_percentage_architecture                          5582 non-null   float64
 3   program_percentage_ethnic_cultural_gender                5582 non-null   float64
 4   program_percentage_communication                         5582 non-null   float64
 5   program_percentage_communications_technology             5582 non-null   float64
 6   program_percentage_computer                              5582 non-null   float64
 7   program_percentage_personal_culinary                     5582 non-null   float64
 8   program_percentage_education

In [ ]:
ridge_residual_df = pd.read_csv(root/"data"/"raw"/"scorecard"/"raw_national_residual_programs.csv")
xgb_residual_df = pd.read_csv(root/"data"/"raw"/"scorecard"/"raw_xgb_national_residual_programs.csv")

In [ ]:
ridge_residual_df['sign_agreement'] = (
    np.sign(ridge_residual_df['1_year_error']) == 
    np.sign(ridge_residual_df['4_year_error'])
) & (
    np.sign(ridge_residual_df['4_year_error']) == 
    np.sign(ridge_residual_df['5_year_error'])
)

print(ridge_residual_df['sign_agreement'].value_counts(normalize=True))

print(ridge_residual_df[['1_year_error','4_year_error','5_year_error']].corr())

sign_agreement
True     0.61211
False    0.38789
Name: proportion, dtype: float64
              1_year_error  4_year_error  5_year_error
1_year_error      1.000000      0.798756      0.714188
4_year_error      0.798756      1.000000      0.816227
5_year_error      0.714188      0.816227      1.000000


In [ ]:
xgb_residual_df['sign_agreement'] = (
    np.sign(xgb_residual_df['1_year_error']) == 
    np.sign(xgb_residual_df['4_year_error'])
) & (
    np.sign(xgb_residual_df['4_year_error']) == 
    np.sign(xgb_residual_df['5_year_error'])
)

print(xgb_residual_df['sign_agreement'].value_counts(normalize=True))

print(xgb_residual_df[['1_year_error','4_year_error','5_year_error']].corr())

sign_agreement
True    1.0
Name: proportion, dtype: float64
              1_year_error  4_year_error  5_year_error
1_year_error      1.000000      0.851905      0.799568
4_year_error      0.851905      1.000000      0.841716
5_year_error      0.799568      0.841716      1.000000


In [ ]:
ridge_residual_df['total_pred'] = ridge_residual_df[
    ["1_year_pred", "4_year_pred", "5_year_pred"]
].sum(axis=1)

ridge_residual_df["total_count"] = (
    ridge_residual_df["1_yr_working_count"] +
    ridge_residual_df["4_yr_working_count"] +
    ridge_residual_df["5_yr_working_count"]
)

k = np.percentile(np.log1p(ridge_residual_df["total_count"]), 75)

ridge_residual_df["weight"] = (
    np.log1p(ridge_residual_df["total_count"]) /
    np.log1p(ridge_residual_df["total_count"] + k)
)

ridge_residual_df["combined_pct_error"] = ridge_residual_df[
    ["1_year_error", "4_year_error", "5_year_error"]
].median(axis=1) / ridge_residual_df['total_pred'] * 3

# consistent_mask = ridge_residual_df['sign_agreement'] == True
# ridge_residual_df = ridge_residual_df[consistent_mask]
# print(f"Consistent schools: {consistent_mask.sum()} of {len(ridge_residual_df)}")

ridge_residual_df.head()

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct,median_score,sign_agreement,total_pred,total_count,weight,combined_pct_error
0,100,3,110422,1,Public,21.0,CA,35.299513,NaN,23,16.0,0.3132,74513.0,0.455561,2.0,20.0,1,NaN,NaN,1036.0,mid,11.132397,11.027100,"Agriculture, General.",California Polytechnic State University-San Lu...,68350.0,61518.918041,6831.081959,84412.0,11.343465,61849.328337,11.032457,22562.671663,36,64786.0,11.078845,44661.282885,10.706862,20124.717115,18.0,28,high,0.450608,0.418702,0.364801,0.352710,0.111040,0.104505,2.0,2.0,10.0,0.0,8.0,8.0,4.618802,0.171067,0.16765,0.352710,True,168029.529263,75.0,0.983097,0.359307
1,100,3,130934,1,Public,24.0,DE,39.187173,NaN,13,14.0,0.4658,39554.0,0.685488,2.0,20.0,1,940.0,NaN,746.0,mid,10.768022,10.889413,"Agriculture, General.",Delaware State University,47478.0,53605.820876,-6127.820876,52676.0,10.871915,54423.909124,10.904559,-1747.909124,24,38873.0,10.568055,39038.464664,10.572303,-165.464664,22.0,28,high,-0.004239,-0.003998,-0.032117,-0.030432,-0.114313,-0.108515,16.0,15.0,18.0,-1.0,3.0,2.0,1.527525,0.056575,0.16765,-0.030432,True,147068.194664,70.0,0.981691,-0.035655
2,100,3,145813,1,Public,132.0,IL,40.509403,NaN,22,16.0,0.8815,67099.0,0.481270,2.0,20.0,1,1113.0,24.0,2439.0,open,11.067279,10.967096,"Agriculture, General.",Illinois State University,64041.0,57936.128323,6104.871677,63600.0,11.060369,59188.291176,10.988479,4411.708824,214,47295.0,10.764160,44425.661994,10.701573,2869.338006,205.0,28,high,0.064587,0.064311,0.074537,0.074227,0.105372,0.104631,11.0,8.0,9.0,-3.0,1.0,-2.0,1.527525,0.056575,0.16765,0.074227,True,161550.081493,551.0,0.998326,0.081926
3,100,3,149222,1,Public,23.0,IL,37.714193,NaN,32,14.0,0.8688,37454.0,0.655376,2.0,22.0,1,1055.0,24.0,3237.0,open,11.051382,10.804913,"Agriculture, General.",Southern Illinois University-Carbondale,63031.0,49262.255760,13768.744240,57596.0,10.961208,50034.114619,10.820460,7561.885381,47,39700.0,10.589106,38819.886275,10.566688,880.113725,22.0,28,high,0.022672,0.021384,0.151135,0.147450,0.279499,0.264632,13.0,4.0,2.0,-9.0,-2.0,-11.0,5.859465,0.217017,0.16765,0.147450,True,138116.256654,92.0,0.986666,0.164250
4,100,3,149772,1,Public,145.0,IL,40.468086,NaN,33,13.0,0.7118,36222.0,0.684009,2.0,21.0,1,NaN,NaN,1807.0,open,10.971709,10.873760,"Agriculture, General.",Western Illinois University,58204.0,52773.247380,5430.752620,58333.0,10.973923,52459.403970,10.867795,5873.596030,160,48509.0,10.789505,38882.312529,10.568295,9626.687471,149.0,28,high,0.247585,0.246045,0.111965,0.111311,0.102907,0.102258,4.0,5.0,11.0,1.0,6.0,7.0,3.785939,0.140220,0.16765,0.111311,True,144114.963878,454.0,0.997908,0.122269


In [ ]:
xgb_residual_df['total_pred'] = xgb_residual_df[
    ["1_year_pred", "4_year_pred", "5_year_pred"]
].sum(axis=1)

xgb_residual_df["total_count"] = (
    xgb_residual_df["1_yr_working_count"] +
    xgb_residual_df["4_yr_working_count"] +
    xgb_residual_df["5_yr_working_count"]
)

k = np.percentile(np.log1p(xgb_residual_df["total_count"]), 75)

xgb_residual_df["weight"] = (
    np.log1p(xgb_residual_df["total_count"]) /
    np.log1p(xgb_residual_df["total_count"] + k)
)

xgb_residual_df["combined_pct_error"] = xgb_residual_df[
    ["1_year_error", "4_year_error", "5_year_error"]
].median(axis=1) / xgb_residual_df['total_pred'] * 3

# consistent_mask = xgb_residual_df['sign_agreement'] == True
# xgb_residual_df = xgb_residual_df[consistent_mask]
# print(f"Consistent schools: {consistent_mask.sum()} of {len(xgb_residual_df)}")

xgb_residual_df.head()

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct,median_score,sign_agreement,total_pred,total_count,weight,combined_pct_error
0,100,3,130934,1,Public,24.0,DE,39.187173,NaN,13,14.0,0.4658,39554.0,0.685488,2.0,20.0,1,940.0,NaN,746.0,mid,10.768022,10.994006,"Agriculture, General.",Delaware State University,47478.0,59516.336,-12038.335937,52676.0,10.871915,59364.930,10.991459,-6688.929687,24,38873.0,10.568055,40220.414,10.602130,-1347.414063,22.0,28,high,-0.033501,-0.031598,-0.112675,-0.106766,-0.202269,-0.192011,18.0,18.0,24.0,0.0,6.0,6.0,3.464102,0.128300,0.186109,-0.106766,True,159101.680,70.0,0.981356,-0.126126
1,100,3,145813,1,Public,132.0,IL,40.509403,NaN,22,16.0,0.8815,67099.0,0.481270,2.0,20.0,1,1113.0,24.0,2439.0,open,11.067279,10.992542,"Agriculture, General.",Illinois State University,64041.0,59429.277,4611.722656,63600.0,11.060369,61214.746,11.022143,2385.253906,214,47295.0,10.764160,42545.203,10.658322,4749.796875,205.0,28,high,0.111641,0.111163,0.038965,0.038803,0.077600,0.077054,8.0,6.0,8.0,-2.0,2.0,0.0,1.154701,0.042767,0.186109,0.077054,True,163189.226,551.0,0.998294,0.084780
2,100,3,149222,1,Public,23.0,IL,37.714193,NaN,32,14.0,0.8688,37454.0,0.655376,2.0,22.0,1,1055.0,24.0,3237.0,open,11.051382,10.850296,"Agriculture, General.",Southern Illinois University-Carbondale,63031.0,51549.410,11481.589844,57596.0,10.961208,53784.330,10.892737,3811.671875,47,39700.0,10.589106,38761.050,10.565171,938.949219,22.0,28,high,0.024224,0.022848,0.070870,0.069142,0.222730,0.210883,13.0,5.0,1.0,-8.0,-4.0,-12.0,6.110101,0.226300,0.186109,0.069142,True,144094.790,92.0,0.986418,0.079358
3,100,3,149772,1,Public,145.0,IL,40.468086,NaN,33,13.0,0.7118,36222.0,0.684009,2.0,21.0,1,NaN,NaN,1807.0,open,10.971709,10.893735,"Agriculture, General.",Western Illinois University,58204.0,53838.008,4365.992187,58333.0,10.973923,53683.992,10.890870,4649.007813,160,48509.0,10.789505,39219.414,10.576927,9289.585938,149.0,28,high,0.236862,0.235389,0.086600,0.086094,0.081095,0.080584,4.0,4.0,7.0,0.0,3.0,3.0,1.732051,0.064150,0.186109,0.086094,True,146741.414,454.0,0.997868,0.095045
4,100,3,157386,0,Public,60.0,KY,38.186768,NaN,33,13.0,0.7721,35407.0,0.689286,2.0,22.0,1,NaN,NaN,533.0,open,10.692785,10.898738,"Agriculture, General.",Morehead State University,44037.0,54108.030,-10071.031250,45255.0,10.720068,54223.120,10.900863,-8968.121094,91,34670.0,10.453630,38868.105,10.567929,-4198.105469,75.0,28,high,-0.108009,-0.106509,-0.165393,-0.163526,-0.186128,-0.182833,25.0,22.0,23.0,-3.0,1.0,-2.0,1.527525,0.056575,0.186109,-0.163526,True,147199.255,226.0,0.995223,-0.182775


In [ ]:
print("XGBoost residual std:", xgb_residual_df["combined_pct_error"].std())
print("Ridge residual std:  ", ridge_residual_df["combined_pct_error"].std())

XGBoost residual std: 0.179119836461294
Ridge residual std:   0.1871690522887098


In [ ]:
join_cols=["unit_id"]
for col in join_cols:
    driver_df[col] = driver_df[col].astype(str).str.strip()
    ridge_residual_df[col] = ridge_residual_df[col].astype(str).str.strip()


### Target Variable Selection

While a composite scoring metric was developed to rank program-level variability, it was not used as the target for the explanatory model.

Instead, the model uses the average  raw percentage error of years 1, 4, and 5 as the target:

  * pct_error = error / predicted

This decision ensures that the model learns directly from observed over- and underperformance, rather than from a derived metric that incorporates additional adjustments (e.g., sample size penalties and variability scaling).

The composite score remains useful for identifying high-variability groups, but the explanatory model focuses on the underlying performance signal.

In [ ]:
# residual_df["combined_pct_error"]=(
#     residual_df[['1_year_error','4_year_error','5_year_error']].sum(axis=1)
#     /
#     residual_df[['1_year_pred','4_year_pred','5_year_pred']].sum(axis=1)
# )

In [ ]:
ridge_residual_df["dollar_error_median"] = ridge_residual_df[
    ["1_year_error", "4_year_error", "5_year_error"]
].median(axis=1)

ridge_residual_df["dollar_pred_median"] = ridge_residual_df[
    ["1_year_pred", "4_year_pred", "5_year_pred"]
].median(axis=1)

ridge_residual_df["row_pct_error"] = (
   ridge_residual_df["dollar_error_median"] / ridge_residual_df["dollar_pred_median"]
)

In [ ]:
school_df = ridge_residual_df.groupby("unit_id", as_index=False).agg(
    combined_pct_error=("row_pct_error", "median"),
    avg_rank_stability=("mean_rank_std_pct", "mean"),
    school_name=("school_name", "first"),
    total_count_1=("1_yr_working_count", "sum"),
    total_count_4=("4_yr_working_count", "sum"),
    total_count_5=("5_yr_working_count", "sum"),
)

school_df["total_count"] = (
    school_df["total_count_1"] +
    school_df["total_count_4"] +
    school_df["total_count_5"]
)

k = np.percentile(np.log1p(school_df["total_count"]), 75)
school_df["weight"] = (
    np.log1p(school_df["total_count"]) /
    np.log1p(school_df["total_count"] + k)
)

In [ ]:
targ="combined_pct_error"
merge_df=driver_df.merge(school_df[[targ,'unit_id','weight']], on=["unit_id"],how="inner")
merge_df.head()

,program_percentage_agriculture,program_percentage_resources,program_percentage_architecture,program_percentage_ethnic_cultural_gender,program_percentage_communication,program_percentage_communications_technology,program_percentage_computer,program_percentage_personal_culinary,program_percentage_education,program_percentage_engineering,program_percentage_engineering_technology,program_percentage_language,program_percentage_family_consumer_science,program_percentage_legal,program_percentage_english,program_percentage_humanities,program_percentage_library,program_percentage_biological,program_percentage_mathematics,program_percentage_military,program_percentage_multidiscipline,program_percentage_parks_recreation_fitness,program_percentage_philosophy_religious,program_percentage_theology_religious_vocation,program_percentage_physical_science,program_percentage_science_technology,program_percentage_psychology,program_percentage_security_law_enforcement,program_percentage_public_administration_social_service,program_percentage_social_science,program_percentage_construction,program_percentage_mechanic_repair_technology,program_percentage_precision_production,program_percentage_transportation,program_percentage_visual_performing,program_percentage_health,program_percentage_business_marketing,program_percentage_history,instructional_expenditure_per_fte,faculty_salary,ft_faculty_rate,program_reporter_programs_offered,student_faculty_ratio,endowment_begin,endowment_end,dolflag,school_name,unit_id,has_endowment,combined_pct_error,weight
0,0.0407,0.0000,0.0136,0.0000,0.0000,0.0542,0.0424,0.0,0.0424,0.1085,0.0203,0.0000,0.0186,0.0,0.0119,0.0661,0.0,0.1424,0.0051,0.0,0.0000,0.0373,0.0000,0.0,0.0237,0.0000,0.0559,0.0644,0.0441,0.0220,0.0,0.0,0.0,0.0,0.0186,0.0000,0.1678,0.0000,7254.0,8699.0,0.6439,NaN,19.0,NaN,NaN,0.0,Alabama A & M University,100654,0,-0.068437,0.999317
1,0.0000,0.0000,0.0000,0.0007,0.0189,0.0000,0.0352,0.0,0.0541,0.0541,0.0000,0.0078,0.0000,0.0,0.0150,0.0303,0.0,0.1489,0.0055,0.0,0.0036,0.0000,0.0036,0.0,0.0176,0.0007,0.0762,0.0358,0.0163,0.0267,0.0,0.0,0.0,0.0,0.0274,0.2111,0.1997,0.0108,17855.0,12612.0,0.7704,NaN,18.0,7.393729e+08,8.589892e+08,0.0,University of Alabama at Birmingham,100663,1,-0.022081,0.999934
2,0.0000,0.0000,0.0000,0.0000,0.0102,0.0000,0.0752,0.0,0.0223,0.3155,0.0122,0.0034,0.0020,0.0,0.0223,0.0000,0.0,0.0616,0.0156,0.0,0.0156,0.0271,0.0007,0.0,0.0427,0.0000,0.0251,0.0000,0.0000,0.0156,0.0,0.0,0.0,0.0,0.0413,0.1043,0.1774,0.0102,9877.0,10639.0,0.6590,NaN,17.0,9.962702e+07,1.137376e+08,1.0,University of Alabama in Huntsville,100706,1,0.041734,0.999451
3,0.0000,0.0000,0.0000,0.0000,0.0511,0.0000,0.0404,0.0,0.0745,0.0128,0.0000,0.0000,0.0000,0.0,0.0085,0.0000,0.0,0.1085,0.0064,0.0,0.0979,0.0128,0.0000,0.0,0.0213,0.0000,0.0638,0.1234,0.0383,0.0298,0.0,0.0,0.0,0.0,0.1043,0.0872,0.1191,0.0000,10723.0,8153.0,0.6477,NaN,15.0,1.186163e+08,1.351989e+08,0.0,Alabama State University,100724,1,-0.029958,0.999299
4,0.0000,0.0068,0.0000,0.0014,0.0950,0.0000,0.0163,0.0,0.0250,0.1002,0.0000,0.0033,0.0628,0.0,0.0098,0.0002,0.0,0.0375,0.0080,0.0,0.0138,0.0392,0.0021,0.0,0.0098,0.0000,0.0517,0.0000,0.0100,0.0900,0.0,0.0,0.0,0.0,0.0241,0.0920,0.2911,0.0100,9728.0,11419.0,0.7904,NaN,19.0,1.369440e+09,1.565892e+09,1.0,The University of Alabama,100751,1,0.002167,0.999954


In [ ]:
model_df=merge_df.copy()
print("scorecard driver schools:", driver_df['unit_id'].nunique())
print("school_df school:", school_df['unit_id'].nunique())
print("model_df schools:", model_df['unit_id'].nunique())

print("model_df columns:")
print(sorted(model_df.columns.tolist()))

scorecard driver schools: 6322
school_df school: 4327
model_df schools: 4327
model_df columns:
['combined_pct_error', 'dolflag', 'endowment_begin', 'endowment_end', 'faculty_salary', 'ft_faculty_rate', 'has_endowment', 'instructional_expenditure_per_fte', 'program_percentage_agriculture', 'program_percentage_architecture', 'program_percentage_biological', 'program_percentage_business_marketing', 'program_percentage_communication', 'program_percentage_communications_technology', 'program_percentage_computer', 'program_percentage_construction', 'program_percentage_education', 'program_percentage_engineering', 'program_percentage_engineering_technology', 'program_percentage_english', 'program_percentage_ethnic_cultural_gender', 'program_percentage_family_consumer_science', 'program_percentage_health', 'program_percentage_history', 'program_percentage_humanities', 'program_percentage_language', 'program_percentage_legal', 'program_percentage_library', 'program_percentage_mathematics', 'pro

In [ ]:
model_df=model_df.drop(columns=[
    # "program_reporter_programs_offered",
    # 'code',
    # 'unit_id',
    # 'school_name'
    ]
)

In [ ]:
program_cols = [c for c in model_df.columns if c.startswith("program_percentage_")]

for c in program_cols:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce").fillna(0)

# STEM
model_df["pct_stem"] = model_df[
    [
        "program_percentage_computer",
        "program_percentage_engineering",
        "program_percentage_engineering_technology",
        "program_percentage_mathematics",
        "program_percentage_physical_science",
        "program_percentage_biological",
        "program_percentage_science_technology"
    ]
].sum(axis=1)

# Business / Econ
model_df["pct_business"] = model_df[
    ["program_percentage_business_marketing"]
].sum(axis=1)

# Health
model_df["pct_health"] = model_df[
    ["program_percentage_health"]
].sum(axis=1)

# Social Sciences
model_df["pct_social_science"] = model_df[
    [
        "program_percentage_psychology",
        "program_percentage_social_science",
        "program_percentage_history",
        "program_percentage_public_administration_social_service"
    ]
].sum(axis=1)

# Humanities
model_df["pct_humanities"] = model_df[
    [
        "program_percentage_english",
        "program_percentage_language",
        "program_percentage_humanities",
        "program_percentage_philosophy_religious",
        "program_percentage_theology_religious_vocation",
        "program_percentage_ethnic_cultural_gender"
    ]
].sum(axis=1)

# Arts & Communication
model_df["pct_arts_comm"] = model_df[
    [
        "program_percentage_visual_performing",
        "program_percentage_communication"
    ]
].sum(axis=1)

# Education
model_df["pct_education"] = model_df[
    ["program_percentage_education"]
].sum(axis=1)

# Trades / Technical
model_df["pct_trades"] = model_df[
    [
        "program_percentage_construction",
        "program_percentage_mechanic_repair_technology",
        "program_percentage_precision_production",
        "program_percentage_transportation"
    ]
].sum(axis=1)

# Services / Consumer
model_df["pct_services"] = model_df[
    [
        "program_percentage_personal_culinary",
        "program_percentage_family_consumer_science",
        "program_percentage_parks_recreation_fitness"
    ]
].sum(axis=1)

# Law / Security
model_df["pct_law_security"] = model_df[
    [
        "program_percentage_legal",
        "program_percentage_security_law_enforcement"
    ]
].sum(axis=1)

# Agriculture / Natural resources
model_df["pct_agriculture"] = model_df[
    [
        # "program_percentage_agriculture",
        "program_percentage_resources"
    ]
].sum(axis=1)

model_df["pct_high_roi"] = (
    model_df["program_percentage_engineering"] +
    model_df["program_percentage_computer"] +
    model_df["program_percentage_health"]
)

model_df["pct_low_roi"] = (
    model_df["program_percentage_education"] +
    model_df["program_percentage_personal_culinary"] +
    model_df["program_percentage_humanities"]
)

model_df["program_hhi"] = (model_df[program_cols] ** 2).sum(axis=1)

model_df["max_program_share"] = model_df[program_cols].max(axis=1)

model_df["high_roi_x_concentration"] = (
    model_df["pct_high_roi"] * model_df["program_hhi"]
)

# model_df = model_df.drop(columns=program_cols)



In [ ]:
display("Repeated columns within instituions",(model_df.groupby("unit_id").nunique() > 1).sum())

'Repeated columns within instituions'

program_percentage_agriculture               0
program_percentage_resources                 0
program_percentage_architecture              0
program_percentage_ethnic_cultural_gender    0
program_percentage_communication             0
                                            ..
pct_high_roi                                 0
pct_low_roi                                  0
program_hhi                                  0
max_program_share                            0
high_roi_x_concentration                     0
Length: 66, dtype: int64

In [ ]:
inst_model_df = model_df.drop_duplicates(subset="unit_id").reset_index(drop=True)
print("inst_model_df shape:", inst_model_df.shape)
display(inst_model_df.head())

inst_model_df shape: (4327, 67)


,program_percentage_agriculture,program_percentage_resources,program_percentage_architecture,program_percentage_ethnic_cultural_gender,program_percentage_communication,program_percentage_communications_technology,program_percentage_computer,program_percentage_personal_culinary,program_percentage_education,program_percentage_engineering,program_percentage_engineering_technology,program_percentage_language,program_percentage_family_consumer_science,program_percentage_legal,program_percentage_english,program_percentage_humanities,program_percentage_library,program_percentage_biological,program_percentage_mathematics,program_percentage_military,program_percentage_multidiscipline,program_percentage_parks_recreation_fitness,program_percentage_philosophy_religious,program_percentage_theology_religious_vocation,program_percentage_physical_science,program_percentage_science_technology,program_percentage_psychology,program_percentage_security_law_enforcement,program_percentage_public_administration_social_service,program_percentage_social_science,program_percentage_construction,program_percentage_mechanic_repair_technology,program_percentage_precision_production,program_percentage_transportation,program_percentage_visual_performing,program_percentage_health,program_percentage_business_marketing,program_percentage_history,instructional_expenditure_per_fte,faculty_salary,ft_faculty_rate,program_reporter_programs_offered,student_faculty_ratio,endowment_begin,endowment_end,dolflag,school_name,unit_id,has_endowment,combined_pct_error,weight,pct_stem,pct_business,pct_health,pct_social_science,pct_humanities,pct_arts_comm,pct_education,pct_trades,pct_services,pct_law_security,pct_agriculture,pct_high_roi,pct_low_roi,program_hhi,max_program_share,high_roi_x_concentration
0,0.0407,0.0000,0.0136,0.0000,0.0000,0.0542,0.0424,0.0,0.0424,0.1085,0.0203,0.0000,0.0186,0.0,0.0119,0.0661,0.0,0.1424,0.0051,0.0,0.0000,0.0373,0.0000,0.0,0.0237,0.0000,0.0559,0.0644,0.0441,0.0220,0.0,0.0,0.0,0.0,0.0186,0.0000,0.1678,0.0000,7254.0,8699.0,0.6439,NaN,19.0,NaN,NaN,0.0,Alabama A & M University,100654,0,-0.068437,0.999317,0.3424,0.1678,0.0000,0.1220,0.0780,0.0186,0.0424,0.0,0.0559,0.0644,0.0000,0.1509,0.1085,0.085876,0.1678,0.012959
1,0.0000,0.0000,0.0000,0.0007,0.0189,0.0000,0.0352,0.0,0.0541,0.0541,0.0000,0.0078,0.0000,0.0,0.0150,0.0303,0.0,0.1489,0.0055,0.0,0.0036,0.0000,0.0036,0.0,0.0176,0.0007,0.0762,0.0358,0.0163,0.0267,0.0,0.0,0.0,0.0,0.0274,0.2111,0.1997,0.0108,17855.0,12612.0,0.7704,NaN,18.0,7.393729e+08,8.589892e+08,0.0,University of Alabama at Birmingham,100663,1,-0.022081,0.999934,0.2620,0.1997,0.2111,0.1300,0.0574,0.0463,0.0541,0.0,0.0000,0.0358,0.0000,0.3004,0.0844,0.124569,0.2111,0.037421
2,0.0000,0.0000,0.0000,0.0000,0.0102,0.0000,0.0752,0.0,0.0223,0.3155,0.0122,0.0034,0.0020,0.0,0.0223,0.0000,0.0,0.0616,0.0156,0.0,0.0156,0.0271,0.0007,0.0,0.0427,0.0000,0.0251,0.0000,0.0000,0.0156,0.0,0.0,0.0,0.0,0.0413,0.1043,0.1774,0.0102,9877.0,10639.0,0.6590,NaN,17.0,9.962702e+07,1.137376e+08,1.0,University of Alabama in Huntsville,100706,1,0.041734,0.999451,0.5228,0.1774,0.1043,0.0509,0.0264,0.0515,0.0223,0.0,0.0291,0.0000,0.0000,0.4950,0.0223,0.158330,0.3155,0.078373
3,0.0000,0.0000,0.0000,0.0000,0.0511,0.0000,0.0404,0.0,0.0745,0.0128,0.0000,0.0000,0.0000,0.0,0.0085,0.0000,0.0,0.1085,0.0064,0.0,0.0979,0.0128,0.0000,0.0,0.0213,0.0000,0.0638,0.1234,0.0383,0.0298,0.0,0.0,0.0,0.0,0.1043,0.0872,0.1191,0.0000,10723.0,8153.0,0.6477,NaN,15.0,1.186163e+08,1.351989e+08,0.0,Alabama State University,100724,1,-0.029958,0.999299,0.1894,0.1191,0.0872,0.1319,0.0085,0.1554,0.0745,0.0,0.0128,0.1234,0.0000,0.1404,0.0745,0.086365,0.1234,0.012126
4,0.0000,0.0068,0.0000,0.0014,0.0950,0.0000,0.0163,0.0,0.0250,0.1002,0.0000,0.0033,0.0628,0.0,0.0098,0.0002,0.0,0.0375,0.0080,0.0,0.0138,0.0392,0.0021,0.0,0.0098,0.0000,0.0517,0.0000,0.0100,0.0900,0.0,0.0,0.0,0.0,0.0241,0.0920,0.2911,0.0100,9728.0,11419.0,0.7904,NaN,19.0,1.369440e+09,1.565892e+09,1.0,The University of Alabama,100751,1,0.00

In [ ]:
display(inst_model_df.describe())
numeric_cols = inst_model_df.select_dtypes(include=[np.number]).columns.tolist()
display("Feature correlation to target variable",
        inst_model_df[numeric_cols].corr()[targ].sort_values())
display(inst_model_df.info())

,program_percentage_agriculture,program_percentage_resources,program_percentage_architecture,program_percentage_ethnic_cultural_gender,program_percentage_communication,program_percentage_communications_technology,program_percentage_computer,program_percentage_personal_culinary,program_percentage_education,program_percentage_engineering,program_percentage_engineering_technology,program_percentage_language,program_percentage_family_consumer_science,program_percentage_legal,program_percentage_english,program_percentage_humanities,program_percentage_library,program_percentage_biological,program_percentage_mathematics,program_percentage_military,program_percentage_multidiscipline,program_percentage_parks_recreation_fitness,program_percentage_philosophy_religious,program_percentage_theology_religious_vocation,program_percentage_physical_science,program_percentage_science_technology,program_percentage_psychology,program_percentage_security_law_enforcement,program_percentage_public_administration_social_service,program_percentage_social_science,program_percentage_construction,program_percentage_mechanic_repair_technology,program_percentage_precision_production,program_percentage_transportation,program_percentage_visual_performing,program_percentage_health,program_percentage_business_marketing,program_percentage_history,instructional_expenditure_per_fte,faculty_salary,ft_faculty_rate,program_reporter_programs_offered,student_faculty_ratio,endowment_begin,endowment_end,dolflag,has_endowment,combined_pct_error,weight,pct_stem,pct_business,pct_health,pct_social_science,pct_humanities,pct_arts_comm,pct_education,pct_trades,pct_services,pct_law_security,pct_agriculture,pct_high_roi,pct_low_roi,program_hhi,max_program_share,high_roi_x_concentration
count,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4321.000000,3072.000000,2912.000000,1409.000000,4202.000000,2.251000e+03,2.251000e+03,4267.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000
mean,0.011191,0.004211,0.001552,0.001072,0.012648,0.003737,0.029479,0.181065,0.025790,0.013613,0.014441,0.002416,0.005930,0.002744,0.006132,0.067669,0.000057,0.024271,0.003424,0.000409,0.011485,0.012945,0.003921,0.005529,0.004854,0.000808,0.026518,0.020064,0.007920,0.018383,0.016018,0.032422,0.016612,0.009413,0.027515,0.248758,0.091439,0.003968,9980.008794,8333.480143,0.607719,5.617459,16.342932,3.669611e+08,3.925047e+08,0.434263,0.520222,0.021425,0.996526,0.090889,0.091439,0.248758,0.056788,0.086738,0.040163,0.025790,0.074465,0.199940,0.022807,0.004211,0.291849,0.274524,0.469220,0.554762,0.165711
std,0.058359,0.018008,0.018414,0.006609,0.044013,0.032788,0.060959,0.369368,0.057262,0.055116,0.048717,0.007959,0.021419,0.026339,0.017487,0.142046,0.001136,0.053937,0.009208,0.008323,0.033631,0.033141,0.048856,0.055035,0.017323,0.007763,0.057192,0.045781,0.027099,0.046114,0.072963,0.116686,0.068208,0.064500,0.099337,0.322664,0.125168,0.009037,15447.418759,2838.348431,0.277490,5.893275,7.694457,2.154191e+09,2.249583e+09,0.495718,0.499649,0.162703,0.005290,0.132947,0.125168,0.322664,0.097986,0.158910,0.111651,0.057262,0.192851,0.362832,0.053301,0.018008,0.319469,0.358955,0.377284,0.337549,0.310759
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0

'Feature correlation to target variable'

ft_faculty_rate             -0.089617
pct_high_roi                -0.079791
program_percentage_health   -0.076195
pct_health                  -0.076195
pct_business                -0.067441
                               ...   
endowment_end                0.068932
endowment_begin              0.070329
pct_low_roi                  0.075232
faculty_salary               0.221839
combined_pct_error           1.000000
Name: combined_pct_error, Length: 65, dtype: float64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4327 entries, 0 to 4326
Data columns (total 67 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   program_percentage_agriculture                           4327 non-null   float64
 1   program_percentage_resources                             4327 non-null   float64
 2   program_percentage_architecture                          4327 non-null   float64
 3   program_percentage_ethnic_cultural_gender                4327 non-null   float64
 4   program_percentage_communication                         4327 non-null   float64
 5   program_percentage_communications_technology             4327 non-null   float64
 6   program_percentage_computer                              4327 non-null   float64
 7   program_percentage_personal_culinary                     4327 non-null   float64
 8   program_percentage_education

None

In [ ]:
def run_school_corr_screen(merge_df, feature_cols, target_col="combined_pct_error", weight_col="weight"):
    
    df_tmp = merge_df[feature_cols + [target_col, weight_col]].copy()
    df_tmp = df_tmp.dropna(subset=[target_col])
    df_tmp = df_tmp.dropna(subset=feature_cols, how="all")
    
    print(f"Rows after dropna: {len(df_tmp)}")
    print(f"Features: {len(feature_cols)}")
    print(f"Target distribution:\n{df_tmp[target_col].describe().round(4)}\n")

    lo, hi = df_tmp[target_col].quantile(0.02), df_tmp[target_col].quantile(0.98)
    df_tmp[target_col] = df_tmp[target_col].clip(lo, hi)

    X = df_tmp[feature_cols].apply(pd.to_numeric, errors="coerce")
    y = df_tmp[target_col]
    w = df_tmp[weight_col]

    corr = X.corrwith(y).abs().sort_values(ascending=False)
    
    print("Top 15 correlations with target:")
    print(corr.head(15).round(4).to_string())
    print(f"\nFeatures with |corr| > 0.10: {(corr > 0.10).sum()}")
    print(f"Features with |corr| > 0.15: {(corr > 0.15).sum()}")
    print(f"Features with |corr| > 0.20: {(corr > 0.20).sum()}")

    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import KFold
    from sklearn.metrics import r2_score
    from sklearn.impute import SimpleImputer

    imputer = SimpleImputer(strategy="median")
    X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rf = RandomForestRegressor(
        n_estimators=100, 
        max_depth=3,
        random_state=42, 
        n_jobs=-1
    )

    fold_r2s = []
    for train_idx, val_idx in kf.split(X_imp):
        X_tr, X_val = X_imp.iloc[train_idx], X_imp.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        w_tr, w_val = w.iloc[train_idx], w.iloc[val_idx]

        rf.fit(X_tr, y_tr, sample_weight=w_tr)
        y_pred = rf.predict(X_val)
        fold_r2s.append(r2_score(y_val, y_pred, sample_weight=w_val))

    print(f"\nRF CV R² (max_depth=3): {np.mean(fold_r2s):.4f} ± {np.std(fold_r2s):.4f}")
    print(f"Fold R²s: {[round(s,3) for s in fold_r2s]}")

    if np.mean(fold_r2s) > 0:
        rf.fit(X_imp, y, sample_weight=w)
        imp_df = pd.DataFrame({
            "feature": feature_cols,
            "importance": rf.feature_importances_
        }).sort_values("importance", ascending=False)
        print("\nTop 10 feature importances:")
        print(imp_df.head(10).to_string(index=False))

    return corr


feature_cols = [c for c in inst_model_df.columns if c not in [
    "unit_id", "school_name", "weight", "combined_pct_error",
    "avg_rank_stability", "total_pred", "median_error",
    "total_count", "total_count_1", "total_count_4", "total_count_5"
]]

corr_results = run_school_corr_screen(inst_model_df, feature_cols)

Rows after dropna: 4327
Features: 63
Target distribution:
count    4327.0000
mean        0.0214
std         0.1627
min        -0.6281
25%        -0.0638
50%         0.0098
75%         0.0946
max         2.0290
Name: combined_pct_error, dtype: float64

Top 15 correlations with target:
faculty_salary                                   0.2213
pct_low_roi                                      0.0993
program_percentage_personal_culinary             0.0845
pct_services                                     0.0827
ft_faculty_rate                                  0.0809
endowment_begin                                  0.0795
program_hhi                                      0.0785
max_program_share                                0.0785
endowment_end                                    0.0779
pct_high_roi                                     0.0770
program_percentage_health                        0.0750
pct_health                                       0.0750
pct_business                               

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV, StratifiedKFold
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
import plotly.express as px

RANDOM_STATE = 42

In [ ]:
pre_y_over = inst_model_df[targ]

percentiles = [0.1, 0.2, 0.3, 0.7, 0.8, 0.9]
values = np.quantile(pre_y_over, percentiles)

fig = px.histogram(pre_y_over, nbins=100, title="Target Distribution with Percentiles")

for p, v in zip(percentiles, values):
    fig.add_vline(
        x=v,
        line_dash="dash",
        annotation_text=f"{int(p*100)}%",
        annotation_position="top"
    )

fig.show()

In [ ]:
inst_model_df[targ].skew()

np.float64(-0.23754546356843945)

In [ ]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, log_loss
from xgboost import XGBClassifier
import optuna



# y_raw = np.log1p(inst_model_df[targ])
# y_raw = y.clip(upper=y.quantile(0.95))
y_raw = inst_model_df[targ].copy()

low = y_raw.quantile(0.3)
high = y_raw.quantile(0.65)

mask = (y_raw <= low) | (y_raw >= high)

X_clf = inst_model_df.drop(columns=["unit_id", targ, "weight",'credential_level','code','avg_rank_stability','school_name'], errors="ignore").loc[mask].copy()
y_clf = (y_raw.loc[mask] >= high).astype(int)
w_clf = inst_model_df['weight'].loc[mask].copy()

X_clf = X_clf.apply(pd.to_numeric, errors="coerce")

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X_clf,
    y_clf,
    w_clf,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y_clf
)


num_cols = X_train.columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols)
])

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 0.5, 3.0),

        # fixed
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist"
    }
    
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('model', XGBClassifier(**params))
    ])

    val_scores = []
    train_scores = []
    
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        w_tr = w_train.iloc[train_idx]

        m = clone(model)
        m.fit(X_tr, y_tr, model__sample_weight=w_tr)
        
        val_scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
        train_scores.append(roc_auc_score(y_tr, m.predict_proba(X_tr)[:, 1]))

    mean_val = np.mean(val_scores)
    mean_train = np.mean(train_scores)
    gap = mean_train - mean_val

    return mean_val - 0.5 * gap

xgb_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)
xgb_study.optimize(objective, n_trials=100)

print("Best trial:")
print("CV AUC:", xgb_study.best_value)
print("Params:", xgb_study.best_params)

best_params = xgb_study.best_params.copy()
best_params.update({
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist"
})

best_xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(**best_params))
])

best_xgb.fit(X_train, y_train, model__sample_weight=w_train)


[I 2026-04-11 17:15:21,088] A new study created in memory with name: no-name-fd28037f-6a41-4521-a503-d5ab9df0be64
[I 2026-04-11 17:15:22,817] Trial 0 finished with value: 0.6199930917493128 and parameters: {'n_estimators': 287, 'max_depth': 6, 'learning_rate': 0.044803926826840625, 'min_child_weight': 9, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 0.00019517224641449495, 'reg_lambda': 2.1423021757741068, 'gamma': 3.005575058716044, 'scale_pos_weight': 2.2701814444901136}. Best is trial 0 with value: 0.6199930917493128.
[I 2026-04-11 17:15:23,861] Trial 1 finished with value: 0.6140275882085485 and parameters: {'n_estimators': 110, 'max_depth': 6, 'learning_rate': 0.060534484680010825, 'min_child_weight': 4, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.6733618039413735, 'reg_alpha': 0.0033205591037519565, 'reg_lambda': 0.042051564509138675, 'gamma': 2.1597250932105787, 'scale_pos_weight': 1.2280728504951048}. Best is trial 0 with value:

Best trial:
CV AUC: 0.6615039292473566
Params: {'n_estimators': 586, 'max_depth': 2, 'learning_rate': 0.009826547602402601, 'min_child_weight': 4, 'subsample': 0.6852850732357625, 'colsample_bytree': 0.8475396952383423, 'reg_alpha': 0.000840034740463124, 'reg_lambda': 0.0030780194501046645, 'gamma': 4.533891065232115, 'scale_pos_weight': 2.384706800480133}


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
train_proba = best_xgb.predict_proba(X_train)[:, 1]
test_proba = best_xgb.predict_proba(X_test)[:, 1]

train_pred = (train_proba >= 0.6845).astype(int)
test_pred = (test_proba >= 0.6945).astype(int)


train_auc = roc_auc_score(y_train, train_proba, sample_weight=w_train)
test_auc = roc_auc_score(y_test, test_proba, sample_weight=w_test)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

train_logloss = log_loss(y_train, train_proba, sample_weight=w_train)
test_logloss = log_loss(y_test, test_proba, sample_weight=w_test)

print("\nXGBClassifier")
print("Train AUC:", round(train_auc, 4))
print("Test AUC:", round(test_auc, 4))
print("Train Accuracy:", round(train_acc, 4))
print("Test Accuracy:", round(test_acc, 4))
print("Train LogLoss:", round(train_logloss, 4))
print("Test LogLoss:", round(test_logloss, 4))

print("\nConfusion Matrix (Test):")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report (Test):")
print(classification_report(y_test, test_pred, digits=4))


XGBClassifier
Train AUC: 0.7838
Test AUC: 0.7262
Train Accuracy: 0.7161
Test Accuracy: 0.6659
Train LogLoss: 0.6502
Test LogLoss: 0.6812

Confusion Matrix (Test):
[[220 169]
 [113 342]]

Classification Report (Test):
              precision    recall  f1-score   support

           0     0.6607    0.5656    0.6094       389
           1     0.6693    0.7516    0.7081       455

    accuracy                         0.6659       844
   macro avg     0.6650    0.6586    0.6587       844
weighted avg     0.6653    0.6659    0.6626       844



In [ ]:
from sklearn.metrics import average_precision_score, f1_score, roc_curve

train_ap = average_precision_score(y_train, train_proba, sample_weight=w_train)
test_ap = average_precision_score(y_test, test_proba, sample_weight=w_test)
train_f1 = f1_score(y_train, train_pred, sample_weight=w_train)
test_f1 = f1_score(y_test, test_pred, sample_weight=w_test)

print("Train AP:", round(train_ap, 4))
print("Test AP:", round(test_ap, 4))
print("Train F1:", round(train_f1, 4))
print("Test F1:", round(test_f1, 4))

Train AP: 0.7814
Test AP: 0.7528
Train F1: 0.7622
Test F1: 0.7084


In [ ]:
fpr, tpr, thresholds = roc_curve(y_train, train_proba)

best_thresh = 0.5
best_f1 = 0

for t in thresholds:
    preds = (train_proba >= t).astype(int)
    f1 = f1_score(y_train, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print(f"Optimal threshold: {best_thresh:.4f}")

test_pred_tuned = (test_proba >= best_thresh).astype(int)

print("\nTuned threshold metrics:")
print(confusion_matrix(y_test, test_pred_tuned))
print(classification_report(y_test, test_pred_tuned, digits=4))

Optimal threshold: 0.6165

Tuned threshold metrics:
[[139 250]
 [ 43 412]]
              precision    recall  f1-score   support

           0     0.7637    0.3573    0.4869       389
           1     0.6224    0.9055    0.7377       455

    accuracy                         0.6528       844
   macro avg     0.6930    0.6314    0.6123       844
weighted avg     0.6875    0.6528    0.6221       844



In [ ]:
import plotly.graph_objects as go
from sklearn.metrics import roc_curve, f1_score, precision_score, recall_score

thresholds = np.linspace(0.01, 0.99, 200)

metrics = {
    "f1": [], "precision": [], "recall": [], "accuracy": []
}

for t in thresholds:
    preds = (train_proba >= t).astype(int)
    metrics["f1"].append(f1_score(y_train, preds, zero_division=0))
    metrics["precision"].append(precision_score(y_train, preds, zero_division=0))
    metrics["recall"].append(recall_score(y_train, preds, zero_division=0))
    metrics["accuracy"].append(accuracy_score(y_train, preds))

best_idx = np.argmax(metrics["f1"])
best_thresh = thresholds[best_idx]

fig = go.Figure()

fig.add_trace(go.Scatter(x=thresholds, y=metrics["f1"],
    name="F1", line=dict(color="#636EFA", width=2)))
fig.add_trace(go.Scatter(x=thresholds, y=metrics["precision"],
    name="Precision", line=dict(color="#EF553B", width=2)))
fig.add_trace(go.Scatter(x=thresholds, y=metrics["recall"],
    name="Recall", line=dict(color="#00CC96", width=2)))
fig.add_trace(go.Scatter(x=thresholds, y=metrics["accuracy"],
    name="Accuracy", line=dict(color="#FFA15A", width=2)))

fig.add_vline(
    x=best_thresh,
    line_dash="dash",
    line_color="white",
    annotation_text=f"Best F1 threshold: {best_thresh:.3f}",
    annotation_position="top right"
)

fig.update_layout(
    title="Threshold vs Classification Metrics (Train)",
    xaxis_title="Threshold",
    yaxis_title="Score",
    legend=dict(orientation="h", y=-0.15),
    template="plotly_dark",
    hovermode="x unified"
)

fig.show()
print(f"\nBest F1 threshold: {best_thresh:.4f}")
print(f"At this threshold — F1: {metrics['f1'][best_idx]:.4f} | "
      f"Precision: {metrics['precision'][best_idx]:.4f} | "
      f"Recall: {metrics['recall'][best_idx]:.4f}")

# apply to test
test_pred_tuned = (test_proba >= best_thresh).astype(int)
print(f"\nTest set with tuned threshold ({best_thresh:.3f}):")
print(confusion_matrix(y_test, test_pred_tuned))
print(classification_report(y_test, test_pred_tuned, digits=4))


Best F1 threshold: 0.6108
At this threshold — F1: 0.7752 | Precision: 0.6556 | Recall: 0.9481

Test set with tuned threshold (0.611):
[[135 254]
 [ 41 414]]
              precision    recall  f1-score   support

           0     0.7670    0.3470    0.4779       389
           1     0.6198    0.9099    0.7373       455

    accuracy                         0.6505       844
   macro avg     0.6934    0.6285    0.6076       844
weighted avg     0.6876    0.6505    0.6177       844

